# Systems and diagrams

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/01_systems.ipynb)

Every plant, controller, filter, noise source, and closed-loop diagram in minilink is a `System`. This tutorial walks through the `System` hierarchy (`System`, `DynamicSystem`, `DiagramSystem`, `StepSystem`, `SampledSystem`, `SampledDiagramSystem`), its main public methods, their arguments, and the types and shapes of their outputs, together with the `Trajectory` record returned by simulation.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `System` and `DynamicSystem`: the equation contract

A continuous-time `DynamicSystem` (subclass of `System`) has a state $x \in \mathbb{R}^n$, an input $u \in \mathbb{R}^m$, and an output $y \in \mathbb{R}^p$:

$$\dot x = f(x, u, t; p), \qquad y = h(x, u, t; p).$$

A static `System` ($n = 0$) has only the output equation. Both are built by the same constructor, which creates the standard ports when asked:

- `System(state_dim=0, *, input_dim=None, output_dim=None, expose_state=False, y_dependencies=None)`
- `DynamicSystem(state_dim, *, input_dim=None, output_dim=None, expose_state=False, y_dependencies=())`

`input_dim` creates the input port `"u"`, `output_dim` the output port `"y"` computed by `h`, `expose_state=True` the output port `"x"`; `y_dependencies` names the input ports `y` reads directly (its feedthrough).

### What a system holds

| Attribute | Type | Meaning |
| --- | --- | --- |
| `sys.n`, `sys.m`, `sys.p` | `int` | State dimension, total input dimension over the input ports, dimension of the `"y"` port |
| `sys.x0` | `ndarray (n,)` | Default initial state of a simulation (read/write) |
| `sys.params` | `dict` | The system's numbers, read when a call passes `params=None` (next tutorial) |
| `sys.name`, `sys.id` | `str`, `str \| None` | Display name (the class name by default) and the key the block takes in a diagram |
| `sys.state` | `VectorSignal` | State descriptor: `.dim`, `.labels`, `.units`, `.nominal_value`, `.lower`, `.upper`, `.box` (a `BoxSet`) |
| `sys.inputs`, `sys.outputs` | `dict[str, InputPort]`, `dict[str, OutputPort]` | Named ports, each a `VectorSignal`; an output port adds `.compute(x, u, t, params)` and `.dependencies` |
| `sys.input_box` | `BoxInputSet` | The input ports' bounds, stacked in port order |
| `sys.u_nominal` | `ndarray (m,)` | The input ports' nominal values, stacked: what an unconnected port reads |
| `sys.solver_info` | `dict` | Solver hints: `"smallest_time_constant"`, `"discontinuous_behavior"` |
| `sys.is_random` | `bool` | Whether the block draws random numbers from a seed in its `params` |
| `sys.traj` | `Trajectory \| None` | The last trajectory `compute_trajectory` produced |

### Equations

| Call | Returns | Meaning |
| --- | --- | --- |
| `sys.f(x, u, t=0, params=None)` | `array (n,)` | State derivative $\dot x$ (`DynamicSystem`) |
| `sys.h(x, u, t=0, params=None)` | `array (p,)` | Primary output $y$, the port `"y"` |
| `sys.outputs[id].compute(x, u, t, params)` | `array (dim,)` | Any output port's signal |
| `sys.ports(u, *port_ids)` | `dict`, `array`, or `tuple` | The stacked input `u` split by input port: every port, one, or several |
| `sys.tf(x, u, t=0, params=None)` | `dict[str, ndarray (4, 4)]` | Body poses by frame name, for rendering (section 6) |

### Verbs

Each verb is a shortcut to a tool that takes the system.

| Call | Returns | Meaning |
| --- | --- | --- |
| `sys.compute_trajectory(tf=10, u=None, *, port=None, t0=0, n_samples=None, dt=None, solver=None, show=False, x0=None, backend="numpy", verbose=False, solver_warnings="warn", record="plant")` | `Trajectory` | Simulate on `[t0, tf]` from `x0` (default `sys.x0`); `u` is `None` (nominal input), a constant, an array `(m, N)`, or a function `u(t)`; `port` names the one input port `u` describes; `record` selects a diagram's recorded signals (section 4). A static `System` has the same call without `solver_warnings` and `record` |
| `sys.compile(*, backend="numpy", verbose=False)` | `DynamicEvaluator` (a diagram: `DiagramEvaluator`; a static system: `StaticEvaluator`) | The system lowered to array functions on NumPy or JAX ([`06_compile_and_evaluate`](06_compile_and_evaluate.ipynb)) |
| `sys.find_equilibrium(x_guess=None, u_bar=None, t=0.0, params=None, *, tol=1e-9)` | `ndarray (n,)` | State $x^\star$ near `x_guess` with $f(x^\star, \bar u, t) = 0$ |
| `sys.jacobian(of, wrt, x_bar=None, u_bar=None, t=0.0, params=None, *, method="auto", eps=1e-6)` | `ndarray` (a `dict` for `wrt="params"`) | $\partial(\text{of}) / \partial(\text{wrt})$ at $(\bar x, \bar u, t)$: `of` is `"f"`, an output port id, or a diagram wire `"block:port"`; `wrt` is `"x"`, `"u"`, an input port id, `"t"`, `"params"`, or a wire |
| `sys.linearize(x_bar=None, u_bar=None, t=0.0, params=None, *, of=None, wrt=None, method="auto", eps=1e-6)` | `LinearizedSystem` | $\dot x = f(\bar x, \bar u) + A\,(x - \bar x) + B\,(u - \bar u)$ in the system's own coordinates, with `.A()`, `.B()`, `.C()`, `.D()`; `.deviation_model()` is the `LTISystem` $\dot{\delta x} = A\,\delta x + B\,\delta u$ |
| `sys.transfer_function(x_bar=None, u_bar=None, t=0.0, params=None, *, of=None, wrt=None, method="auto", eps=1e-6, minimal=None)` | `TransferFunction` | Transfer function of the linearization, from input `wrt` to output `of` |
| `sys.discretize(dt, *, solver="rk4", substeps=1)` | `SampledSystem` | The sampled-data model at period `dt`, input held (section 5) |
| `sys.compute_cost(cost, *, of=None, traj=None)` | `float` | The cost $J$ of a trajectory ([`04_costs`](04_costs.ipynb)) |
| `sys.plot_trajectory(traj=None, *, signals=None, renderer="matplotlib", show=True)` | `PlotResult` | Time plot of a trajectory (default `sys.traj`) |
| `sys.plot_phase_plane(traj=None, *, x_axis=0, y_axis=None, renderer="matplotlib", show=True, **kwargs)` | `PlotResult` | Vector field of two states, with the trajectory over it |
| `sys.plot_diagram(file_name=None, show=True, *, expand=True)` | `graphviz.Digraph` | The block diagram |
| `sys.plot_bode(...)`, `sys.plot_nyquist(...)`, `sys.plot_pzmap(...)`, `sys.plot_root_locus(...)`, `sys.plot_step_response(...)` | `PlotResult` | Linear-analysis plots at an operating point (the arguments of `linearize` first) |
| `sys.plot_control_law(...)`, `sys.plot_input_output_map(...)`, `sys.plot_cost(cost, *, of=None, traj=None, renderer="matplotlib", show=True)` | `PlotResult` | A static map over two axes; the cost along a trajectory |
| `sys.animate(...)`, `sys.render(...)`, `sys.game(...)` | — | Graphics (section 6) |

In [ ]:
import numpy as np

from minilink import Pendulum

plant = Pendulum()
print(plant)

### Inspecting dimensions, state metadata, and ports

`plant.state` and `plant.inputs` / `plant.outputs` hold the labels, units, nominal values, and bounds:

In [ ]:
plant.n, plant.m, plant.p, plant.x0

In [ ]:
plant.state.labels, plant.state.units, plant.state.box

In [ ]:
list(plant.inputs), list(plant.outputs), plant.input_box, plant.u_nominal

### Calling the equations `f`, `h`, and `tf`

`f` and `h` are pure functions of their arguments: they never mutate the system. They keep the array backend (NumPy arrays in give NumPy arrays out; JAX arrays in give JAX arrays out).

In [ ]:
x = np.array([0.5, 0.0])
u = np.array([0.0])

dx = plant.f(x, u, t=0.0)
dx.shape, dx

In [ ]:
y = plant.h(x, u, t=0.0)
y.shape, y

In [ ]:
poses = plant.tf(x, u, t=0.0)
list(poses), poses["link"].shape

### Analysis shortcuts: `find_equilibrium`, `jacobian`, and `linearize`

The analysis verbs work around an operating point $(\bar x, \bar u)$, by default the nominal state and input:

In [ ]:
x_eq = plant.find_equilibrium(x_guess=np.array([3.0, 0.0]))
x_eq

In [ ]:
df_dx = plant.jacobian("f", "x", x_bar=x_eq, u_bar=np.zeros(1))
df_du = plant.jacobian("f", "u", x_bar=x_eq, u_bar=np.zeros(1))
df_dx.shape, df_du.shape, df_dx

In [ ]:
lin = plant.linearize(x_bar=x_eq, u_bar=np.zeros(1))
lin.A(), lin.B()

## 2. Simulating a system: `Trajectory` and `Simulator`

`compute_trajectory` returns a `Trajectory`, the frozen record `Trajectory(t, x, u, signals={}, roles={})` of every simulation and plan:

| Attribute / call | Returns | Meaning |
| --- | --- | --- |
| `traj.t` | `ndarray (N,)` | Sample times |
| `traj.x` | `ndarray (n, N)` | State samples, a column per sample |
| `traj.u` | `ndarray (m, N)` | Input samples |
| `traj.n`, `traj.m`, `traj.n_samples` | `int` | State dimension, input dimension, sample count $N$ |
| `traj.t0`, `traj.tf` | `float` | First and last sample times |
| `traj.signals` | read-only mapping `str -> ndarray (dim, N)` | Extra recorded signals |
| `traj.roles` | read-only mapping | A diagram run's layout: each block's state slice and the signal that records its input, read by `of` |
| `traj.signal_names` | `tuple[str, ...]` | Every channel name: `"x"`, `"u"`, then the extra signals |
| `traj.has_signal(name)` | `bool` | Whether `name` is a channel |
| `traj.signal(name)` | `ndarray (dim, N)` | A channel by name (`"x"` and `"u"` included) |
| `traj.at(t, name="x")` | `ndarray (dim,)` | A channel at time `t`, linear between the samples |
| `traj.resample(*, t_new=None, n_samples=None)` | `Trajectory` | The trajectory interpolated on another time grid |
| `traj.with_signals(values)` | `Trajectory` | A copy with the signals of the dict `values` added, each `(dim, N)` |
| `traj.with_cost(cost, *, discount_rate=0.0, params=None, t0=0.0)` | `Trajectory` | A copy with the signals `"dJ"` and `"J"`, each `(1, N)` ([`04_costs`](04_costs.ipynb)) |
| `traj.of(role)` | `Trajectory` | One block's own `(t, x, u)` from a diagram run (section 4) |
| `traj.save(path)`, `Trajectory.load(path)` | `None`, `Trajectory` | Write to and read from a compressed `.npz` file |

In [ ]:
plant.x0 = np.array([1.0, 0.0])
traj = plant.compute_trajectory(tf=5.0)
print(traj)

In [ ]:
traj.t.shape, traj.x.shape, traj.u.shape, traj.t0, traj.tf, traj.n_samples

In [ ]:
traj.at(2.5), traj.at(2.5, "u")

In [ ]:
coarse_traj = traj.resample(n_samples=21)
coarse_traj.t.shape, coarse_traj.x.shape

In [ ]:
forced_traj = plant.compute_trajectory(tf=5.0, u=lambda t: np.array([1.5 * np.sin(2.0 * t)]))
forced_traj.u.shape, forced_traj.x[:, -1]

In [ ]:
plant.plot_trajectory(traj);

In [ ]:
plant.plot_phase_plane(traj);

### `Simulator`: the tool behind `compute_trajectory`

`compute_trajectory` builds a `Simulator`, which compiles the system once, chooses a solver and a time grid, and integrates. Holding the simulator runs it again without a recompile:

| Call | Returns | Meaning |
| --- | --- | --- |
| `Simulator(sys, tf=10, *, x0=None, t0=0, n_samples=None, dt=None, solver=None, verbose=False, backend="numpy", solver_warnings="warn")` | `Simulator` | Compile `sys`, choose the solver and the grid on `[t0, tf]` |
| `sim.solve(*, u=None, port=None, params=None, record=())` | `Trajectory` | One run; `params` is `None` (the numbers taken at the build) or the complete tree of `sim.params`, edited |
| `sim.params` | `dict` | The numbers the simulator runs on, a copy |
| `sim.t`, `sim.dt`, `sim.n_samples` | `ndarray (N,)`, `float`, `int` | The time grid |
| `sim.solver_mode`, `sim.backend` | `str`, `str` | The solver chosen and the array backend |
| `sim.evaluator` | compiled evaluator | The compiled system |

`solver` is `"rk4"` or `"euler"` (fixed step, on the grid) or an adaptive SciPy preset: `"scipy"` (RK45), `"scipy_stiff"` (Radau), `"scipy_max"` and `"scipy_ultra"` (DOP853), `"scipy_lsoda"`. With `solver=None` the simulator picks `"euler"` for a system that declares discontinuous behavior, `"rk4"` when a block holds a signal over a sample period (and on the JAX backend with a long uniform grid given explicitly), and `"scipy"` otherwise. A fixed-step solver with no `dt` or `n_samples` steps at a tenth of `sys.solver_info["smallest_time_constant"]`.

In [ ]:
from minilink import Simulator

sim = Simulator(plant, tf=5.0, dt=0.01, solver="rk4")
sim.solver_mode, sim.n_samples

In [ ]:
longer = sim.params
longer["l"] = 2.0
sim.solve().x[:, -1], sim.solve(params=longer).x[:, -1]

## 3. Writing a `DynamicSystem` and its ports

A model subclasses `DynamicSystem`, declares its sizes in `super().__init__`, sets `self.params`, and writes `f` (and `h`). Further ports are declared with the port builders:

| Call | Returns | Meaning |
| --- | --- | --- |
| `sys.add_input_port(id, *, dim=None, nominal_value=None, labels=None, units=None, lower=None, upper=None)` | `None` | Add the input port `sys.inputs[id]` |
| `sys.add_output_port(id, *, dim=None, function=None, dependencies=(), nominal_value=None, labels=None, units=None, lower=None, upper=None)` | `None` | Add the output port `sys.outputs[id]`, computed by `function(x, u, t, params)`; `dependencies` names the input ports it reads directly |
| `sys.rename_ports(**names)` | the system | Rename ports in place, `old_id=new_id` (`sys.rename_ports(y="theta")`) |

In [ ]:
from minilink import DynamicSystem
from minilink.core.backends import array_module


class SpringMass(DynamicSystem):
    def __init__(self):
        super().__init__(2, input_dim=1, output_dim=1)
        self.params = {"m": 1.0, "k": 4.0, "b": 0.4}
        self.state.labels = ["position", "velocity"]
        self.state.units = ["m", "m/s"]
        self.add_output_port(
            "energy",
            dim=1,
            function=self.energy,
            labels=["mechanical energy"],
            units=["J"],
        )

    def f(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)

        m = params["m"]
        k = params["k"]
        b = params["b"]

        q = x[0]
        v = x[1]
        force = u[0]

        # m q̈ = F − b q̇ − k q
        a = (force - b * v - k * q) / m
        dx = xp.stack([v, a])

        return dx

    def h(self, x, u, t=0, params=None):
        y = x[0:1]

        return y

    def energy(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)

        m = params["m"]
        k = params["k"]

        q = x[0]
        v = x[1]

        E = 0.5 * m * v**2 + 0.5 * k * q**2
        out = xp.stack([E])

        return out


spring = SpringMass()
spring.x0 = np.array([1.0, 0.0])
spring_traj = spring.compute_trajectory(tf=6.0)
list(spring.outputs), spring.outputs["energy"].compute(spring.x0, np.zeros(1), 0.0, None)

In [ ]:
spring.plot_trajectory(spring_traj);

## 4. `DiagramSystem`: wiring blocks into a composite system

A `DiagramSystem` is itself a `DynamicSystem`: its state is the stack of its subsystems' states, and every simulation, analysis, and compilation verb works on it unchanged.

### Building a diagram

| Call | Returns | Meaning |
| --- | --- | --- |
| `controller @ plant` | `DiagramSystem` | Closed loop: the standard feedback ports wired, an `Error` block `e = r - y` inserted for an error-driven controller. The function form is `closed_loop(controller, plant, *, reference=True, disturbance=False, noise=False, filter=None, observer=None)` |
| `first >> second` | `DiagramSystem` | Series: the primary output of `first` into the primary input of `second` |
| `a + b` | `DiagramSystem` | The two blocks side by side, not connected |
| `DiagramSystem()` | `DiagramSystem` | An empty diagram, for explicit wiring |
| `diagram.add_subsystem(sys, sys_id=None)` | `None` | Add a block under `sys_id` (default: the id the operators would give it, from its `id` or its name) |
| `diagram.connect(source_sys_id, source_port_id, target_sys_id, target_port_id)` | `None` | Wire an output port to an input port; the source id `"input"` is the diagram's own input ports |
| `diagram.disconnect(target_sys_id, target_port_id)` | `None` | Open an input again |
| `diagram.connect_new_output_port(source_sys_id, source_port_id, output_port_id, dependencies=None, *, nominal_value=None, labels=None, units=None, lower=None, upper=None)` | `None` | Put an inside output on the diagram's edge under `output_port_id` |
| `diagram.expose(*names)` | the diagram | Put inside ports on the edge by name: the open inputs of that name become one edge input, the one inside output of that name an edge output |
| `diagram.autowire(*, strict=False)` | the diagram | Connect each open input to the one output with its name and size |

The operators also compose step and sampled systems (section 5). Neither operand is modified: the result is a new diagram that shares its blocks.

### Reading a diagram

| Attribute / call | Returns | Meaning |
| --- | --- | --- |
| `diagram.subsystems` | `dict[str, System]` | The blocks by id |
| `diagram.connections` | `dict[str, dict[str, tuple[str, str] \| None]]` | For each block and input port, its source `(sys_id, port_id)`, `None` when open; the entry `"output"` holds the edge outputs |
| `diagram.state_index` | `dict[str, tuple[int, int]]` | Each block's `(start, stop)` in the diagram's state |
| `diagram.params` | `dict[str, dict]` | `{sys_id: subsystem.params}`, live (next tutorial) |
| `diagram.subsystem_id(subsystem)` | `str` | The id of a block instance (or an id, checked) |
| `diagram.of(block)` | `System` | The block by its id, or by a path into a nested diagram, `"inner/plant"` |
| `diagram.subsystem_signal(subsystem, port_id)` | `str` | The signal name `"sys_id:port_id"` of a block's output port |
| `diagram.reconstruct_internal_signals(traj, ports=None, params=None)` | `Trajectory` | `traj` with internal signals computed from its samples and attached, after a run that did not record them (`ports=None`: every output port) |
| `diagram.port_samples(traj, name, params=None)` | `ndarray (dim, N)` | One internal signal, `"block:port"` or `"block:u"`, along `traj` |
| `diagram.compile(*, backend="numpy", verbose=False, flatten=False)` | `DiagramEvaluator` | The diagram lowered to one flat function |

In [ ]:
from minilink import PID

pd = PID(Kp=10.0, Ki=4.0, Kd=2.0)
diagram = pd @ spring
print(diagram)

In [ ]:
list(diagram.subsystems), diagram.connections, diagram.state_index

In [ ]:
diagram.subsystem_id(spring), diagram.of("plant") is spring

In [ ]:
diagram.plot_diagram();

Explicit wiring, for a topology the operators do not reach: a `Step` source into the spring's force, and the spring's two outputs on the diagram's edge (`connect_new_output_port` names the block, since both blocks have a port `"y"`; `expose` finds the one port `"energy"`).

In [ ]:
from minilink import DiagramSystem, Step

manual = DiagramSystem()
manual.add_subsystem(Step(final_value=2.0), "source")
manual.add_subsystem(spring, "plant")
manual.connect("source", "y", "plant", "u")
manual.connect_new_output_port("plant", "y", "y")
manual.expose("energy")
print(manual)

### What a diagram run records

`compute_trajectory(record=...)` selects the internal signals kept in `traj.signals`:

| `record=` | Signals kept |
| --- | --- |
| `"plant"` (default) | The plant's whole input `"plant:u"`, when the diagram has a block `plant` |
| `None` | None (the fastest run) |
| a sequence of names | Exactly those: `"block:port"` an output port, `"block:u"` a block's whole input, a nested block by its path |
| `"all"` | Every output port of every block |

A sequence replaces the default, so it names `"plant:u"` to keep it. `traj.of("plant")` is then the plant's own `Trajectory`: its state slice and the input it received.

In [ ]:
loop_traj = diagram.compute_trajectory(tf=4.0, record=("plant:u", "plant:energy"))
loop_traj.signal_names, loop_traj.signal("plant:u").shape

In [ ]:
plant_only_traj = loop_traj.of("plant")
plant_only_traj.x.shape, plant_only_traj.u.shape

In [ ]:
diagram.plot_trajectory(loop_traj);

## 5. Discrete and sampled-data systems: `StepSystem`, `SampledSystem`, `SampledDiagramSystem`

Discrete-time systems form a sibling algebra beside the continuous one:

- **`StepSystem`** ($n \ge 1$): a recurrence on an integer tick $k$,
  $$x_{k+1} = \mathrm{step}(x_k, u_k, k; p), \qquad y_k = h(x_k, u_k, k; p).$$
  Its constructor is `StepSystem(state_dim, *, input_dim=None, output_dim=None, expose_state=False, y_dependencies=())`.
- **`SampledSystem(source, sample_period, *, solver="rk4", substeps=1, kind="model")`**: a view of `source` on a clock of period `sample_period` seconds, its input held over each period. It is built by a verb, never by hand:
  - `plant.discretize(dt, *, solver="rk4", substeps=1)` is a discrete-time **model** (`kind="model"`), $\dot x = f$ integrated over each period;
  - `block % dt` puts a block on the computer's clock (`kind="law"`): a continuous law becomes its Euler difference equation, a static block is sampled and held, a `StepSystem` gets its clock.
- **`StepDiagramSystem`** and **`SampledDiagramSystem`**: `>>`, `+` and `@` between step systems give a `StepDiagramSystem`; between systems on a clock, a `SampledDiagramSystem` on the base period (the greatest common divisor of its blocks' periods). `(controller % dt) @ plant` is the one place the two algebras meet: the controller ticks every `dt` while the continuous plant is integrated underneath.

| Call | Returns | Meaning |
| --- | --- | --- |
| `step_sys.step(x, u, k=0, params=None)` | `array (n,)` | Next state $x_{k+1}$ |
| `step_sys.h(x, u, k=0, params=None)` | `array (p,)` | Output $y_k$ |
| `step_sys.sample_period` | `float \| None` | The clock's period in seconds; `None` on a tick law |
| `step_sys.compute_rollout(n_steps, u=None, *, x0=None, t0=0.0, dt=None, record="plant", backend="numpy", show=False, verbose=False)` | `Trajectory` | `n_steps` ticks, `n_steps + 1` samples at `t = k` (on a clock: `t0 + k * sample_period`); `u` is `None`, a constant, an array `(m, n_steps)`, or a function `u(k)` |
| `step_sys.compute_trajectory(tf=10.0, u=None, *, x0=None, t0=0.0, dt=None, solver=None, backend="numpy", record="plant", show=False, verbose=False)` | `Trajectory` | On a clock with continuous time underneath: the same run with the plant's substeps kept. A tick law refuses it |
| `step_sys.jacobian(of, wrt, x_bar=None, u_bar=None, k=0, params=None, *, method="auto", eps=1e-6)` | `ndarray` | As on a continuous system, with `of="step"` the recurrence |
| `step_sys.plot_rollout(traj=None, *, signals=None, renderer="matplotlib", show=True)` | `PlotResult` | Plot of a rollout's sampled signals (default: the last rollout) |
| `sampled.source`, `sampled.solver`, `sampled.substeps`, `sampled.kind` | `System`, `str`, `int`, `str` | What the clock was applied to, and how the period is integrated |
| `sampled.update(x, u, t, params=None)` | `array (n,)` | The seconds face: $x(t + \Delta t)$ from $x(t)$ and the held $u(t)$; `step(x, u, k)` is `update(x, u, k * sample_period)` |
| `sampled.update_path(x, u, t, params=None)` | `array (substeps, n)` | The state at the end of every substep of `update` |
| `sampled.with_steps(substeps=None, solver=None)` | `SampledSystem` | The same view with another integration step |
| `sampled_loop.compile(*, backend="numpy", verbose=False, substeps=None, solver=None)` | `SampledDiagramEvaluator` | The loop lowered to one map on the base tick |

In [ ]:
from minilink import StepSystem


class MovingAverage(StepSystem):
    def __init__(self, alpha=0.3):
        super().__init__(1, input_dim=1, output_dim=1)
        self.params = {"alpha": alpha}

    def step(self, x, u, k=0, params=None):
        params = self.params if params is None else params

        alpha = params["alpha"]

        x_next = (1.0 - alpha) * x + alpha * u

        return x_next


avg = MovingAverage()
x_next = avg.step(np.array([0.0]), np.array([1.0]), k=0)
rollout = avg.compute_rollout(15, u=np.ones((1, 15)))
x_next, rollout.t.shape, rollout.x.shape

In [ ]:
avg.plot_rollout(rollout);

A `SampledSystem` has both faces, `update(x, u, t)` in seconds and `step(x, u, k)` on the tick:

In [ ]:
sampled_spring = spring.discretize(0.05)
x_start = np.array([1.0, 0.0])
(
    sampled_spring.kind,
    sampled_spring.sample_period,
    sampled_spring.update(x_start, np.zeros(1), t=0.0),
    sampled_spring.step(x_start, np.zeros(1), k=0),
)

A sampled loop on a continuous plant has two runs: `compute_rollout` gives the ticks, `compute_trajectory` the same run with the plant's substeps kept. Both record the plant's held input by default, so `of("plant")` is the plant's own run:

In [ ]:
digital_loop = (pd % 0.02) @ spring
ticks = digital_loop.compute_rollout(200)
digital_traj = digital_loop.compute_trajectory(tf=4.0)
type(digital_loop).__name__, digital_loop.sample_period, ticks.t.shape, digital_traj.of("plant")

In [ ]:
digital_loop.plot_trajectory(digital_traj);

## 6. The graphics contract: `tf`, geometry, `render`, and `animate`

A system describes how it looks; it holds no rendering engine. Three hooks return dictionaries keyed by frame name, and the renderers read them:

| Hook | Returns | Meaning |
| --- | --- | --- |
| `sys.tf(x, u, t=0, params=None)` | `dict[str, ndarray (4, 4)]` | World pose of each named frame (forward kinematics) |
| `sys.get_kinematic_geometry()` | `dict[str, list]` | Static shapes attached to each frame, built once; the base method delegates to the attribute `sys.skin`, a callable `(sys) -> dict` or `None`, so assigning a skin swaps the look without touching `f` or `tf` |
| `sys.get_dynamic_geometry(x, u, t=0, params=None)` | `dict[str, list]` | Shapes rebuilt at each frame (force and torque arrows) |
| `sys.camera_target`, `sys.camera_plot_axes`, `sys.camera_scale`, `sys.camera_follow_frame` | `ndarray (3,)`, `tuple`, `float \| None`, `str \| None` | Camera hints: the point looked at, the two world axes of a 2D view, the half-width of the view (`None` fits the geometry), a frame to follow |

A diagram stacks its blocks' frames and geometry. The verbs that draw:

| Call | Returns | Meaning |
| --- | --- | --- |
| `sys.render(x, u, t, *, is_3d=False, renderer="matplotlib", camera=None, overlays=None)` | `None` | Draw one frame |
| `sys.animate(traj=None, *, speed=1.0, is_3d=False, html=None, renderer="matplotlib", native=True, scene_title=None, show=True, save=False, file_name="Animation", camera=None, overlays=None)` | the renderer's animation | Play a trajectory (default `sys.traj`); `renderer` is `"matplotlib"`, `"meshcat"`, `"plotly"` or `"pygame"`; `save=True` writes a GIF (matplotlib) or an HTML page (meshcat) |
| `sys.game(...)` | — | A live keyboard or joystick session (`DynamicSystem`) |

In [ ]:
shapes = plant.get_kinematic_geometry()
arrows = plant.get_dynamic_geometry(x, u, t=0.0)
list(shapes), len(shapes["link"]), list(arrows), plant.camera_target, plant.camera_plot_axes

## Summary

| Class | State and time | Equations | Runs |
| --- | --- | --- | --- |
| `System` ($n=0$) | Stateless | `h(x, u, t, params)` | `compute_trajectory` |
| `DynamicSystem` | Continuous $x(t)$ | `f(x, u, t, params)`, `h(x, u, t, params)` | `compute_trajectory` |
| `DiagramSystem` | Stack of its blocks' states | `f`, `h` | `compute_trajectory(record=...)`, `traj.of("plant")` |
| `StepSystem` | Discrete $x_k$ | `step(x, u, k, params)`, `h(x, u, k, params)` | `compute_rollout` |
| `SampledSystem` | $x_k$ at $t_k = k\,\Delta t$ | `update(x, u, t, params)`, `step(x, u, k, params)` | `compute_rollout` |
| `SampledDiagramSystem` | Clocked blocks on a continuous plant | `step`, `h` | `compute_rollout`, `compute_trajectory` |

Next, [`02_parameters`](02_parameters.ipynb): how every number of a system, diagram, set, field, distribution, cost, and problem lives in `.params`.